In [ ]:
import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATASET_PATH = ROOT_DIR / "dataset" / "raw" / "career_navigator_synthetic.csv"
PROCESSED_DIR = ROOT_DIR / "dataset" / "processed"
MODEL_DIR = ROOT_DIR / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (15000, 49)


In [2]:
target_columns = [
    "placement_status",
    "career_readiness",
    "recommended_career_domain",
    "employability_score"
]

X = df.drop(columns=["student_id"] + target_columns)

y_placement = df["placement_status"]
y_readiness = df["career_readiness"]
y_domain = df["recommended_career_domain"]
y_employability = df["employability_score"]

print("Input feature shape:", X.shape)

print("\nTarget shapes:")
print("Placement:", y_placement.shape)
print("Career readiness:", y_readiness.shape)
print("Career domain:", y_domain.shape)
print("Employability:", y_employability.shape)

assert X.shape[1] == 44
assert not set(target_columns).intersection(X.columns)
assert "student_id" not in X.columns

Input feature shape: (15000, 44)

Target shapes:
Placement: (15000,)
Career readiness: (15000,)
Career domain: (15000,)
Employability: (15000,)


In [3]:
numerical_features = X.select_dtypes(include=np.number).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical columns:")
print(numerical_features)

print("\nCategorical columns:")
print(categorical_features)

Numerical features: 41
Categorical features: 3

Numerical columns:
['study_year', 'academic_score_10th', 'academic_score_12th', 'undergraduate_cgpa', 'current_cgpa', 'backlogs', 'attendance_percentage', 'academic_projects', 'major_projects', 'practical_skill_score', 'research_experience', 'training_hours', 'programming_score', 'problem_solving_score', 'aptitude_score', 'technical_knowledge_score', 'communication_score', 'teamwork_score', 'leadership_score', 'adaptability_score', 'internship_count', 'internship_duration_months', 'certifications_count', 'hackathon_participation', 'hackathon_wins', 'github_projects', 'resume_score', 'mock_interview_score', 'interview_preparation_hours', 'interest_ai_ml', 'interest_data_science', 'interest_software_dev', 'interest_cybersecurity', 'interest_cloud', 'interest_uiux', 'skill_ai_ml', 'skill_data_science', 'skill_software_dev', 'skill_cybersecurity', 'skill_cloud', 'skill_uiux']

Categorical columns:
['education_level', 'degree', 'specialization

C:\Users\DHRUV\AppData\Local\Temp\ipykernel_6516\4221681670.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [4]:
X_train, X_temp, y_train_placement, y_temp_placement = train_test_split(
    X,
    y_placement,
    test_size=0.30,
    random_state=42,
    stratify=y_placement
)

X_val, X_test, y_val_placement, y_test_placement = train_test_split(
    X_temp,
    y_temp_placement,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_placement
)

train_indices = X_train.index
val_indices = X_val.index
test_indices = X_test.index

y_train_readiness = y_readiness.loc[train_indices]
y_val_readiness = y_readiness.loc[val_indices]
y_test_readiness = y_readiness.loc[test_indices]

y_train_domain = y_domain.loc[train_indices]
y_val_domain = y_domain.loc[val_indices]
y_test_domain = y_domain.loc[test_indices]

y_train_employability = y_employability.loc[train_indices]
y_val_employability = y_employability.loc[val_indices]
y_test_employability = y_employability.loc[test_indices]

print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))

assert len(X_train) == 10500
assert len(X_val) == 2250
assert len(X_test) == 2250

Training records: 10500
Validation records: 2250
Testing records: 2250


In [5]:
print("PLACEMENT DISTRIBUTION")

for name, target in [
    ("Training", y_train_placement),
    ("Validation", y_val_placement),
    ("Testing", y_test_placement)
]:
    print(f"\n{name}:")
    print((target.value_counts(normalize=True) * 100).round(2))

print("\nCAREER READINESS DISTRIBUTION")

for name, target in [
    ("Training", y_train_readiness),
    ("Validation", y_val_readiness),
    ("Testing", y_test_readiness)
]:
    print(f"\n{name}:")
    print((target.value_counts(normalize=True) * 100).round(2))

print("\nCAREER DOMAIN DISTRIBUTION")

for name, target in [
    ("Training", y_train_domain),
    ("Validation", y_val_domain),
    ("Testing", y_test_domain)
]:
    print(f"\n{name}:")
    print((target.value_counts(normalize=True) * 100).round(2))

PLACEMENT DISTRIBUTION

Training:
placement_status
0    63.54
1    36.46
Name: proportion, dtype: float64

Validation:
placement_status
0    63.56
1    36.44
Name: proportion, dtype: float64

Testing:
placement_status
0    63.56
1    36.44
Name: proportion, dtype: float64

CAREER READINESS DISTRIBUTION

Training:
career_readiness
Moderate    50.22
Low         24.94
High        24.84
Name: proportion, dtype: float64

Validation:
career_readiness
Moderate    49.07
High        26.13
Low         24.80
Name: proportion, dtype: float64

Testing:
career_readiness
Moderate    49.91
Low         25.47
High        24.62
Name: proportion, dtype: float64

CAREER DOMAIN DISTRIBUTION

Training:
recommended_career_domain
Software Development                          36.88
Cybersecurity                                 19.65
UI/UX Design                                  12.38
Artificial Intelligence & Machine Learning    10.62
Data Science                                  10.27
Cloud Computing          

In [6]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    sparse_threshold=0
)

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed!")

print("\nProcessed shapes:")
print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Testing:", X_test_processed.shape)

Preprocessing completed!

Processed shapes:
Training: (10500, 65)
Validation: (2250, 65)
Testing: (2250, 65)


In [7]:
readiness_encoder = LabelEncoder()
domain_encoder = LabelEncoder()

y_train_readiness_encoded = readiness_encoder.fit_transform(
    y_train_readiness
)

y_val_readiness_encoded = readiness_encoder.transform(
    y_val_readiness
)

y_test_readiness_encoded = readiness_encoder.transform(
    y_test_readiness
)

y_train_domain_encoded = domain_encoder.fit_transform(
    y_train_domain
)

y_val_domain_encoded = domain_encoder.transform(
    y_val_domain
)

y_test_domain_encoded = domain_encoder.transform(
    y_test_domain
)

print("Career readiness classes:")
print(dict(zip(
    readiness_encoder.classes_,
    readiness_encoder.transform(readiness_encoder.classes_)
)))

print("\nCareer domain classes:")
print(dict(zip(
    domain_encoder.classes_,
    domain_encoder.transform(domain_encoder.classes_)
)))

Career readiness classes:
{'High': np.int64(0), 'Low': np.int64(1), 'Moderate': np.int64(2)}

Career domain classes:
{'Artificial Intelligence & Machine Learning': np.int64(0), 'Cloud Computing': np.int64(1), 'Cybersecurity': np.int64(2), 'Data Science': np.int64(3), 'Software Development': np.int64(4), 'UI/UX Design': np.int64(5)}


In [8]:
y_train_employability_array = y_train_employability.to_numpy(dtype=np.float32)
y_val_employability_array = y_val_employability.to_numpy(dtype=np.float32)
y_test_employability_array = y_test_employability.to_numpy(dtype=np.float32)

print("Regression targets prepared.")

print("Training mean:", y_train_employability_array.mean())
print("Validation mean:", y_val_employability_array.mean())
print("Testing mean:", y_test_employability_array.mean())

Regression targets prepared.
Training mean: 55.373917
Validation mean: 55.569046
Testing mean: 55.241337


In [9]:
print("FINAL PREPROCESSING CHECK")

print("Training shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Testing shape:", X_test_processed.shape)

print("\nMissing values in processed data:")

print("Training:", np.isnan(X_train_processed).sum())
print("Validation:", np.isnan(X_val_processed).sum())
print("Testing:", np.isnan(X_test_processed).sum())

assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_val_processed).all()
assert np.isfinite(X_test_processed).all()

assert X_train_processed.shape[1] == X_val_processed.shape[1]
assert X_train_processed.shape[1] == X_test_processed.shape[1]

print("\nAll preprocessing checks passed!")

FINAL PREPROCESSING CHECK
Training shape: (10500, 65)
Validation shape: (2250, 65)
Testing shape: (2250, 65)

Missing values in processed data:
Training: 0
Validation: 0
Testing: 0

All preprocessing checks passed!


In [10]:
arrays = {
    "X_train": X_train_processed,
    "X_val": X_val_processed,
    "X_test": X_test_processed,

    "y_train_placement": y_train_placement.to_numpy(),
    "y_val_placement": y_val_placement.to_numpy(),
    "y_test_placement": y_test_placement.to_numpy(),

    "y_train_readiness": y_train_readiness_encoded,
    "y_val_readiness": y_val_readiness_encoded,
    "y_test_readiness": y_test_readiness_encoded,

    "y_train_domain": y_train_domain_encoded,
    "y_val_domain": y_val_domain_encoded,
    "y_test_domain": y_test_domain_encoded,

    "y_train_employability": y_train_employability_array,
    "y_val_employability": y_val_employability_array,
    "y_test_employability": y_test_employability_array
}

for name, array in arrays.items():
    np.save(PROCESSED_DIR / f"{name}.npy", array)

joblib.dump(
    preprocessor,
    MODEL_DIR / "preprocessor.joblib"
)

joblib.dump(
    readiness_encoder,
    MODEL_DIR / "readiness_encoder.joblib"
)

joblib.dump(
    domain_encoder,
    MODEL_DIR / "domain_encoder.joblib"
)

feature_names = preprocessor.get_feature_names_out().tolist()

with open(PROCESSED_DIR / "processed_feature_names.json", "w") as file:
    json.dump(feature_names, file, indent=4)

print("All processed datasets and preprocessing artifacts saved!")

All processed datasets and preprocessing artifacts saved!


In [11]:
summary = {
    "original_records": len(df),
    "original_features": X.shape[1],
    "processed_features": X_train_processed.shape[1],
    "training_records": len(X_train),
    "validation_records": len(X_val),
    "testing_records": len(X_test),
    "random_state": 42,
    "split_ratio": {
        "training": 0.70,
        "validation": 0.15,
        "testing": 0.15
    },
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "readiness_classes": readiness_encoder.classes_.tolist(),
    "domain_classes": domain_encoder.classes_.tolist()
}

with open(PROCESSED_DIR / "preprocessing_summary.json", "w") as file:
    json.dump(summary, file, indent=4)

print("Preprocessing summary saved.")
print("\nPREPROCESSING PIPELINE COMPLETED SUCCESSFULLY!")

Preprocessing summary saved.

PREPROCESSING PIPELINE COMPLETED SUCCESSFULLY!


In [12]:
from pathlib import Path
import numpy as np
import joblib
import json

print("=" * 60)
print("CAREER NAVIGATOR - FINAL PREPROCESSING VERIFICATION")
print("=" * 60)

# 1. Verify processed arrays
for name, array in arrays.items():
    print(f"{name:28} {array.shape}")

# 2. Verify missing and infinite values
print("\nNUMERICAL DATA CHECK")

for name in ["X_train", "X_val", "X_test"]:
    array = arrays[name]
    print(
        f"{name}: NaN = {np.isnan(array).sum()}, "
        f"Infinite = {np.isinf(array).sum()}"
    )
    assert np.isfinite(array).all()

# 3. Verify target dimensions
print("\nTARGET DIMENSIONS")

for target_name in [
    "y_train_placement",
    "y_train_readiness",
    "y_train_domain",
    "y_train_employability"
]:
    print(target_name, arrays[target_name].shape)

# 4. Verify preprocessing artifacts
artifacts = [
    MODEL_DIR / "preprocessor.joblib",
    MODEL_DIR / "readiness_encoder.joblib",
    MODEL_DIR / "domain_encoder.joblib",
    PROCESSED_DIR / "preprocessing_summary.json",
    PROCESSED_DIR / "processed_feature_names.json"
]

print("\nSAVED ARTIFACTS")

for artifact in artifacts:
    print(f"{artifact.name}: {'FOUND' if artifact.exists() else 'MISSING'}")
    assert artifact.exists()

# 5. Verify split sizes
assert len(arrays["X_train"]) == 10500
assert len(arrays["X_val"]) == 2250
assert len(arrays["X_test"]) == 2250

# 6. Verify feature dimensions
assert arrays["X_train"].shape[1] == arrays["X_val"].shape[1]
assert arrays["X_train"].shape[1] == arrays["X_test"].shape[1]

print("\nALL PREPROCESSING VERIFICATION CHECKS PASSED!")

CAREER NAVIGATOR - FINAL PREPROCESSING VERIFICATION
X_train                      (10500, 65)
X_val                        (2250, 65)
X_test                       (2250, 65)
y_train_placement            (10500,)
y_val_placement              (2250,)
y_test_placement             (2250,)
y_train_readiness            (10500,)
y_val_readiness              (2250,)
y_test_readiness             (2250,)
y_train_domain               (10500,)
y_val_domain                 (2250,)
y_test_domain                (2250,)
y_train_employability        (10500,)
y_val_employability          (2250,)
y_test_employability         (2250,)

NUMERICAL DATA CHECK
X_train: NaN = 0, Infinite = 0
X_val: NaN = 0, Infinite = 0
X_test: NaN = 0, Infinite = 0

TARGET DIMENSIONS
y_train_placement (10500,)
y_train_readiness (10500,)
y_train_domain (10500,)
y_train_employability (10500,)

SAVED ARTIFACTS
preprocessor.joblib: FOUND
readiness_encoder.joblib: FOUND
domain_encoder.joblib: FOUND
preprocessing_summary.json: FOUND
